# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [6]:
#create citations table

#store first row as header
citations_header = rddCitations.first()

#remove headers
citations = rddCitations.filter(lambda x: x != citations_header)

#split each row by comma delimiter
citations = citations.map(lambda x: x.split(","))

#cast both values to int and store as a key value pair
citations = citations.map(lambda x: (int(x[0]),int(x[1])))

In [7]:
#do the same for patents table
header = rddPatents.first()

patents = rddPatents.filter(lambda x: x != header)

patents = patents.map(lambda x: x.split(","))

#select item 0 and 5 as patent number (cast as int) and postate
patents = patents.map(lambda x: (int(x[0]),(x[5])))



In [8]:
#join patents to citations on citing
temp = citations.join(patents)

In [9]:
#reorder output to keep citing patents and move cited into key
temp2 = temp.map(lambda x: (x[1][0], (x[0],x[1][1])))

#join patents to citations again on cited
temp2 = temp2.join(patents)

In [10]:
#reorder the values so it is citing, citing state, cited, cited state
reconstructed = temp2.map(lambda x: (x[1][0][0], x[1][0][1], x[0], x[1][1]))

In [11]:
#filter where citing and cited states are equal and remove when either state is blank
same_state = reconstructed.filter(lambda x: x[1] == x[3]).filter(lambda x: x[1] != '""').filter(lambda x: x[3] != '""')

In [12]:
#get counts by remapping each citing patent a value of 1 and then reducing by key across the RDD
counts = same_state.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)

In [13]:
#make new patents table for final join
patents_data_header = rddPatents.first()

patents_data = rddPatents.filter(lambda x: x != header)

patents_data = patents_data.map(lambda x: x.split(","))

#store patents_data with patent as key and remaining columns as value
patents_data = patents_data.map(lambda x: (int(x[0]),x[1:]))

In [14]:
#join counts to reconstructed patents
final = patents_data.join(counts)

#sort final by counts and re-map the output to match format specified in assignment
top10 = final.sortBy(lambda x: x[1][1],ascending=False).map(lambda x: [x[0]] + x[1][0] + [x[1][1]])

#print top 10
top10.take(10)

[[5959466,
  '1999',
  '14515',
  '1997',
  '"US"',
  '"CA"',
  '5310',
  '2',
  '',
  '326',
  '4',
  '46',
  '159',
  '0',
  '1',
  '',
  '0.6186',
  '',
  '4.8868',
  '0.0455',
  '0.044',
  '',
  '',
  125],
 [5983822,
  '1999',
  '14564',
  '1998',
  '"US"',
  '"TX"',
  '569900',
  '2',
  '',
  '114',
  '5',
  '55',
  '200',
  '0',
  '0.995',
  '',
  '0.7201',
  '',
  '12.45',
  '0',
  '0',
  '',
  '',
  103],
 [6008204,
  '1999',
  '14606',
  '1998',
  '"US"',
  '"CA"',
  '749584',
  '2',
  '',
  '514',
  '3',
  '31',
  '121',
  '0',
  '1',
  '',
  '0.7415',
  '',
  '5',
  '0.0085',
  '0.0083',
  '',
  '',
  100],
 [5952345,
  '1999',
  '14501',
  '1997',
  '"US"',
  '"CA"',
  '749584',
  '2',
  '',
  '514',
  '3',
  '31',
  '118',
  '0',
  '1',
  '',
  '0.7442',
  '',
  '5.1102',
  '0',
  '0',
  '',
  '',
  98],
 [5958954,
  '1999',
  '14515',
  '1997',
  '"US"',
  '"CA"',
  '749584',
  '2',
  '',
  '514',
  '3',
  '31',
  '116',
  '0',
  '1',
  '',
  '0.7397',
  '',
  '5.181',
 